In [ ]:
import numpy as np

class neural_network:
  def __init__(self, neurons1, neurons2, input_dim,  output_dim):
    self.neurons1 = neurons1
    self.neurons2 = neurons2

    self.input_dim = input_dim
    self.output_dim = output_dim

    self.layer1_weight, self.layer1_bias, self.layer2_weight, self.layer2_bias, self.layer3_weight, self.layer3_bias = self.create_network(self.input_dim, self.neurons1, self.neurons2, self.output_dim)
    
  def get_weight_vector(self, input, neurons):
    weight_vector = np.random.uniform(low=-1.0, high=1.0, size=(input, neurons))
    return weight_vector
  def get_bias_vector(self, neurons):
    bias_vector = np.random.uniform(low=-1.0, high=1.0, size=(1,neurons))
    return bias_vector

  def relu(self, x):
      return np.maximum(0, x)
  def relu_derivative(self, x):
      return np.where(x > 0, 1.0, 0.0)
  def sigmoid(self, x):
      return 1 / (1 + np.exp(-x))
  def softmax(self,x):
    if x.ndim == 1:
        x = x.reshape(1, -1)
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e_x / np.sum(e_x, axis=1, keepdims=True)


  def create_network(self, input_neurons, layer1_neurons, layer2_neurons, output_neurons):
    layer1_weight = self.get_weight_vector(input_neurons, layer1_neurons)
    layer1_bias = self.get_bias_vector(layer1_neurons)

    layer2_weight = self.get_weight_vector(layer1_neurons, layer2_neurons)
    layer2_bias = self.get_bias_vector(layer2_neurons)

    layer3_weight = self.get_weight_vector(layer2_neurons, output_neurons)
    layer3_bias = self.get_bias_vector(output_neurons)

    return layer1_weight, layer1_bias, layer2_weight, layer2_bias, layer3_weight, layer3_bias




  def forward_pass(self, input_data):
    
    layer1_linear = (input_data@self.layer1_weight) + self.layer1_bias
    layer1_output = self.relu(layer1_linear)
    layer2_linear = (layer1_output@self.layer2_weight) + self.layer2_bias
    layer2_output = self.relu(layer2_linear)
    layer3_output = (layer2_output@self.layer3_weight) + self.layer3_bias
    final_output = self.softmax(layer3_output)

    return layer1_linear, layer1_output, layer2_linear, layer2_output, final_output


  def reset_weights(self):
    self.layer1_weight, self.layer1_bias, self.layer2_weight, self.layer2_bias, self.layer3_weight, self.layer3_bias = self.create_network(self.input_dim, self.neurons1, self.neurons2, self.output_dim)

  def one_hot_encoder(self, labels): 
    one_hot_labels = np.zeros((labels.shape[0], self.output_dim))
    for index, number in enumerate(labels):
       one_hot_labels[index][number]= 1.0
    return one_hot_labels
     
  def flatten_input(self, array):
    new_shape = array.shape[:-2] + (array.shape[-2] * array.shape[-1],)
    flattened = array.reshape(new_shape)
    return flattened
  
  def fit(self, input_data, data_labels , learning_rate, batch_size, epochs):
    n_samples = batch_size * (input_data.shape[0] // batch_size)
    input_data = input_data[:n_samples]
    data_labels = data_labels[:n_samples]

    n_batches = input_data.shape[0] // batch_size
    input_data = self.flatten_input(input_data)
    data_labels = self.one_hot_encoder(data_labels)


    for j in range(epochs):
        indices = np.arange(input_data.shape[0])
        np.random.shuffle(indices)
        input_data = input_data[indices]
        data_labels = data_labels[indices]
        for i in range((n_batches)):

            batch = input_data[i*batch_size:(i+1)*batch_size]
            batch_labels = data_labels[i*batch_size:(i+1)*batch_size]
            # 1. Forward Pass (für den GESAMTEN Batch auf einmal)
            # xor_input hat Shape (batch_size, in_dim)
            batch = input_data[i*batch_size:(i+1)*batch_size]
            batch_labels = data_labels[i*batch_size:(i+1)*batch_size]
            # 1. Forward Pass (für den GESAMTEN Batch auf einmal)
            # xor_input hat Shape (batch_size, in_dim)
            l1_linear, l1_out, l2_linear, l2_out, output = self.forward_pass(batch)
            
            # 2. Backpropagation Layer 2 (Output-Schicht)
            # Shape: (batch_size, out_dim)
            error_output = (output - batch_labels)/batch_size  # Durchschnitt über den Batch
            
            # Gradienten für Layer 2
            # Transponieren von l1_out: (hidden_dim, batch_size) @ (batch_size, out_dim) -> (hidden_dim, out_dim)
            grad_w3 = np.dot(l2_out.T, error_output)
            grad_b3 = np.sum(error_output, axis=0)  # Summe über alle Zeilen (den Batch)
            
            # 3. Backpropagation Layer 1 (Hidden-Schicht)
            # (batch_size, out_dim) @ (out_dim, hidden_dim) -> (batch_size, hidden_dim)
            error_hidden_2 = np.dot(error_output, self.layer3_weight.T) * self.relu_derivative(l2_linear)
            error_hidden_1 = np.dot(error_hidden_2, self.layer2_weight.T) * self.relu_derivative(l1_linear)
            # Gradienten für Layer 1
            # Transponieren von input_data: (in_dim, batch_size) @ (batch_size, hidden_dim) -> (in_dim, hidden_dim)
            grad_w2 = np.dot(l1_out.T, error_hidden_2)  # Durchschnitt über den Batch
            grad_b2 = np.sum(error_hidden_2, axis=0) # Summe über den Batch
            
            error_hidden_1 = np.dot(error_hidden_2, self.layer2_weight.T) * self.relu_derivative(l1_linear)
            
            grad_w1 = np.dot(batch.T, error_hidden_1)
            grad_b1 = np.sum(error_hidden_1, axis=0)
            
            
            # 4. Gewichte und Biases aktualisieren
            self.layer3_weight -= learning_rate * grad_w3
            self.layer3_bias   -= learning_rate * grad_b3
            self.layer2_weight -= learning_rate * grad_w2
            self.layer2_bias   -= learning_rate * grad_b2
            self.layer1_weight -= learning_rate * grad_w1
            self.layer1_bias   -= learning_rate * grad_b1



  # --- ERGEBNISSE NACH DEM TRAINING ---
  def predict(self, input_data):
    from matplotlib import pyplot as plt
    flattened_input = self.flatten_input(input_data)
    _, _, _, _, pred = self.forward_pass(flattened_input)
    print(f"Input: {input_data} -> Vorhersage: {np.argmax(pred)}, mit der Warscheinlichkeit: {pred}")
    plt.imshow(input_data, cmap='gray')  # cmap='gray' sorgt für die korrekte Schwarz-Weiß-Darstellung
    plt.axis('off')                     # Blendat die Achsenbeschriftung für eine sauberere Optik aus
    plt.show()
    


  def evaluate(self, test_input_data, test_output_data):
    test_input_data = self.flatten_input(test_input_data)
    test_output_data = self.one_hot_encoder(test_output_data)
   
    _, _, _, _, pred = self.forward_pass(test_input_data)

        
    # Categorical Cross-Entropy Loss
    eps = 1e-15
    pred_clipped = np.clip(pred, eps, 1 - eps)
    loss = -np.mean(np.sum(test_output_data * np.log(pred_clipped), axis=1))
    
  
    return loss



: 

In [ ]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 820.8 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 79.0 MB/s eta 0:00:00:00:0100:01


In [ ]:

import numpy as np

import random as rd
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

from tensorflow.keras.datasets import mnist # type: ignore

# Loads directly as 3D arrays
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalize to 0.0 - 1.0
X_train = X_train / 255.0
X_test = X_test / 255.0

train_test_split = 0.8
X_test =  X_test[0:int(X_test.shape[0]*0.1)]
y_test = y_test[0:int(y_test.shape[0]*0.1)]

print(X_train.shape)  
print(X_test.shape)
#input_train_data= input_data[0:int(input_data.shape[0]*train_test_split)]
#output_train_data = output_data[0:int(output_data.shape[0]*train_test_split)]

#input_test_data = input_data[int(input_data.shape[0]*train_test_split):]
#output_test_data = output_data[int(output_data.shape[0]*train_test_split):]

neuronen_exp = 3
starting_neurons = 64
average_loss = []
print("Starting Grid Search for Neurons and Learning Rate")
for i in range(1,neuronen_exp+1):
  model = neural_network(neurons1=int(starting_neurons*(2**(i-1))), neurons2=int(starting_neurons*(2**(i-3))), input_dim=784, output_dim=10)
  for j in range(1,10):
    model.reset_weights()
    model.fit(X_train, y_train, learning_rate=0.3*(0.8**j), batch_size=200, epochs=25)
    evaluation = model.evaluate(X_train, y_train)
    average_loss.append(evaluation)
    print(f"Neurons: {starting_neurons*(2**(i-1)), starting_neurons*(2**(i-3))}, Learning Rate: {0.3*(0.8**j)}, Average Loss: {average_loss[-1]}, Accuracy: {1-average_loss[-1]}")

print(average_loss)
print(min(average_loss))
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


# 1. Die 1D-Liste in eine 2D-Matrix umwandeln (6 Zeilen für Neuronen, 9 Spalten für Learning Rate)
loss_matrix = np.array(average_loss).reshape(neuronen_exp, 9)

# 2. Beschriftungen aus deinen Schleifenbereich definieren (range(1,9) und range(1,10))
neuron_counts = [2**(i+5) for i in range(1, neuronen_exp+1)]  # [2, 4, 8, 16, 32, 64, 128, 256]
learning_rates = [
    0.1 * j for j in range(1, 10)
]  # [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# 3. Heatmap zeichnen
plt.figure(figsize=(10, 6))
sns.heatmap(
    loss_matrix,
    annot=True,
    fmt=".4f",
    xticklabels=learning_rates,
    yticklabels=neuron_counts,
    cmap="viridis_r",
    cbar_kws={"label": "Average Loss"},
)

plt.title("Grid Search: Average Loss", fontsize=14)
plt.xlabel("Learning Rate", fontsize=12)
plt.ylabel("Anzahl Neuronen (Hidden Layer)", fontsize=12)
plt.tight_layout()
plt.show()










/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
(60000, 28, 28)
(1000, 28, 28)
Starting Grid Search for Neurons and Learning Rate


KeyboardInterrupt: 